# ***Simulador de satélite***
## 1. Carga de librerías

Este bloque importa dependencias básicas.
En este caso usamos solo librerías estándar, pero queda preparado para escalar.

In [ ]:
import random
import numpy as np

## 2. Configuración global

Aquí definimos todos los parámetros del sistema.

Esto permite modificar el comportamiento sin tocar la lógica.

In [ ]:
class Config:
    TIME_STEPS = 60
    MAX_ENERGY = 40
    MAX_MEMORY = 100

    CLOUD_PROB = 0.3

    ENERGY_COST_CAPTURE = 20
    MEMORY_GAIN_CAPTURE = 20

## 3. Definición del entorno

El entorno contiene los objetivos (targets) y la incertidumbre (nubes).

In [ ]:
class Target:
   def __init__(self, id, time_window, value, accessible=True):
        self.id = id
        self.time_window = time_window
        self.value = value
        self.done = False
        self.accessible = accessible


class Environment:
    def __init__(self):
        self.targets = [
            Target("A", (5, 10), 10, True),
            Target("B", (10, 20), 20, True),
            Target("C", (12, 18), 25, True),
            Target("D", (35, 40), 25, True),
        ]

    def is_cloudy(self):
        return random.random() < Config.CLOUD_PROB

## 4. Modelo del satélite

Este bloque representa el estado interno del agente.

In [ ]:
class Satellite:
    def __init__(self):
        self.energy = Config.MAX_ENERGY
        self.memory = 0
        self.time = 0

    def consume_energy(self, amount):
        self.energy = max(0, self.energy - amount)

    def add_memory(self, amount):
        self.memory = min(Config.MAX_MEMORY, self.memory + amount)

    def recharge(self, amount=5):
        self.energy = min(Config.MAX_ENERGY, self.energy + amount)

    def downlink(self):
        self.memory = max(0, self.memory - 50)

    def __str__(self):
        return f"[t={self.time}] Energy={self.energy:.1f}, Memory={self.memory}"

## 5. Planificador (arquitectura deliberativa)

Este bloque implementa la planificación global.

In [ ]:
class Planner:
    def plan(self, targets):
        # Ordenar por valor (heurística simple)
        return sorted(targets, key=lambda t: -t.value)

## 6. Control reactivo

Responde rápidamente a condiciones críticas.

In [ ]:
class ReactiveController:
    def act(self, sat):
        if sat.energy < 20:
            print("⚡ REACTIVO: energía baja → recargando")
            sat.recharge()
            return True

        if sat.memory > 80:
            print("📡 REACTIVO: memoria alta → downlink")
            sat.downlink()
            return True

        return False

## 7. MPC (replanificación local)

Ajusta el plan dinámicamente según el estado actual.

In [ ]:
class MPC:
    def adjust_plan(self, plan, sat):
        new_plan = []
        for t in plan:
            if not t.done and sat.time <= t.time_window[1]:
                new_plan.append(t)
        return new_plan

## 8. Simulador (integración de todo)

Este bloque integra todos los componentes.

In [ ]:
class Simulator:
    def __init__(self):
        self.env = Environment()
        self.sat = Satellite()

        self.planner = Planner()
        self.reactive = ReactiveController()
        self.mpc = MPC()

        self.plan = self.planner.plan(self.env.targets)

    def step(self):
        print("\n----------------------")
        print(self.sat)

        # 1. REACTIVO (prioridad)
        if self.reactive.act(self.sat):
            return

        # 2. MPC
        self.plan = self.mpc.adjust_plan(self.plan, self.sat)

        # 3. EJECUCIÓN DEL PLAN
        action_taken = False

        for target in self.plan:
            start, end = target.time_window

            if start <= self.sat.time <= end and not target.done:

                if self.env.is_cloudy():
                    print(f"🌫️ Nubes en {target.id} → cancelar")
                    action_taken = True
                    break

                print(f"📸 Capturando {target.id}")
                self.sat.consume_energy(Config.ENERGY_COST_CAPTURE)
                self.sat.add_memory(Config.MEMORY_GAIN_CAPTURE)
                target.done = True

                action_taken = True
                break

        if not action_taken:
            print("🔄 Idle / carga solar")
            self.sat.recharge(2)

    def run(self):
        print("🛰️ INICIO SIMULACIÓN")

        for t in range(Config.TIME_STEPS):
            self.sat.time = t
            self.step()

        self.results()

    def results(self):
        print("\n📊 RESULTADOS:")
        for t in self.env.targets:
            print(f"{t.id}: {'✔️' if t.done else '❌'}")

## 9. Ejecución del sistema

Este bloque ejecuta la simulación completa.

In [ ]:
sim = Simulator()
sim.run()